In [1]:
from src.data.dataset_loader import load_saved_datasets

train_ds, val_ds, test_ds = load_saved_datasets()

print("Train dataset spec:", train_ds.element_spec)
print("\nVal dataset spec:", val_ds.element_spec)
print("\nTest dataset spec:", test_ds.element_spec)

Train dataset spec: ({'attention_mask': TensorSpec(shape=(None, 128), dtype=tf.int32, name=None), 'input_ids': TensorSpec(shape=(None, 128), dtype=tf.int32, name=None)}, TensorSpec(shape=(None, 6), dtype=tf.float32, name=None))

Val dataset spec: ({'attention_mask': TensorSpec(shape=(None, 128), dtype=tf.int32, name=None), 'input_ids': TensorSpec(shape=(None, 128), dtype=tf.int32, name=None)}, TensorSpec(shape=(None, 6), dtype=tf.float32, name=None))

Test dataset spec: ({'attention_mask': TensorSpec(shape=(None, 128), dtype=tf.int32, name=None), 'input_ids': TensorSpec(shape=(None, 128), dtype=tf.int32, name=None)}, TensorSpec(shape=(None, 6), dtype=tf.float32, name=None))


In [2]:
train_batches = sum(1 for _ in train_ds)
val_batches = sum(1 for _ in val_ds)
test_batches = sum(1 for _ in test_ds)

print(f"Train batches: {train_batches} (~{train_batches * 32} samples)")
print(f"Val batches: {val_batches} (~{val_batches * 32} samples)")
print(f"Test batches: {test_batches} (~{test_batches * 32} samples)")

Train batches: 4488 (~143616 samples)
Val batches: 499 (~15968 samples)
Test batches: 1995 (~63840 samples)


In [3]:
for features, labels in train_ds.take(1):
    print("input_ids shape:", features['input_ids'].shape)
    print("attention_mask shape:", features['attention_mask'].shape)
    print("labels shape:", labels.shape)
    print("\nSample input_ids (first row):")
    print(features['input_ids'][0])
    print("\nSample attention_mask (first row):")
    print(features['attention_mask'][0])
    print("\nSample labels (first row):")
    print(labels[0])

input_ids shape: (32, 128)
attention_mask shape: (32, 128)
labels shape: (32, 6)

Sample input_ids (first row):
tf.Tensor(
[  101  1000  2204  2391  1998 25380  1012  2008  1005  1055  2025  2129
  2009  2246  2012  1996  2051  1012  2002  1005  1055  2908  2185  1012
  2005  2085  1012  2021 21660  2135  7827  1996  7065  8743  6462  2097
  2074  4929  2041  2115  4344  1998 25975  2115 10627  1012  2022  2062
 25022 11890 18163  5051  6593  1012  3189  1996  5248  1998  3357  2067
  1012  2320  2002  1005  1055  9411  2007  1010  2057  2064  2131  2006
  2007 26157 10086  2015  1012 25022  7869 21138  1005  1005  1005  1000
   102     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0], shape=(128,), dtype=int32)

Sample attention_mask (first row):
tf.Tensor(
[1 1 1 1 1 1 1 1 1 1 1

In [4]:
from src.data.preprocess import get_tokenizer

tokenizer = get_tokenizer()

for features, labels in train_ds.take(1):
    sample_ids = features['input_ids'][0].numpy()
    decoded_text = tokenizer.decode(sample_ids, skip_special_tokens=True)
    print("Decoded text (first sample in batch):")
    print(decoded_text)
    print("\nCorresponding labels:", labels[0].numpy())
    print("Label columns order:", ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"])

Decoded text (first sample in batch):
" good point and apologies. that ' s not how it looked at the time. he ' s gone away. for now. but relentlessly pressing the revert button will just wear out your finger and fray your nerves. be more circumspect. report the behavior and step back. once he ' s dealt with, we can get on with constructive edits. cierekim ' ' ' "

Corresponding labels: [0. 0. 0. 0. 0. 0.]
Label columns order: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [5]:
import numpy as np

for features, labels in train_ds.take(1):
    attention_masks = features['attention_mask'].numpy()
    real_token_counts = attention_masks.sum(axis=1)
    print("Real token counts per sample (out of 128):")
    print(real_token_counts)
    print(f"\nMean real tokens: {real_token_counts.mean():.1f}")

Real token counts per sample (out of 128):
[ 85 128  42 128  22  33  20 115  56  82  68  19   7  21  36  29  24  91
  36  24  42  30   7 128 128  30  28  18  76  25  18  15]

Mean real tokens: 50.3


In [6]:
def get_label_distribution(dataset, name):
    all_labels = []
    for _, labels in dataset:
        all_labels.append(labels.numpy())
    all_labels = np.concatenate(all_labels, axis=0)
    label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
    print(f"\n{name} set label distribution:")
    for i, col in enumerate(label_cols):
        pos_count = all_labels[:, i].sum()
        pct = pos_count / len(all_labels) * 100
        print(f"  {col}: {int(pos_count)} ({pct:.2f}%)")

get_label_distribution(train_ds, "Train")
get_label_distribution(val_ds, "Validation")


Train set label distribution:
  toxic: 13779 (9.60%)
  severe_toxic: 1441 (1.00%)
  obscene: 7615 (5.30%)
  threat: 436 (0.30%)
  insult: 7096 (4.94%)
  identity_hate: 1257 (0.88%)

Validation set label distribution:
  toxic: 1515 (9.49%)
  severe_toxic: 154 (0.97%)
  obscene: 834 (5.23%)
  threat: 42 (0.26%)
  insult: 781 (4.89%)
  identity_hate: 148 (0.93%)


In [7]:
get_label_distribution(test_ds, "Test")


Test set label distribution:
  toxic: 6090 (9.54%)
  severe_toxic: 367 (0.57%)
  obscene: 3691 (5.78%)
  threat: 211 (0.33%)
  insult: 3427 (5.37%)
  identity_hate: 712 (1.12%)
